# B. Acid-Base Pairs
Investigate the gas-phase acidity of __one__ other conjugate acid–base pairs from the list below (see Workshop Instructor for allocation of one of the acid-base pairs below) by calculating the standard Gibbs free energy for acid ionisation at 298.15 K.

* formic acid
* acetic acid
* benzoic acid
* phenol
* methanol




First we need to install the DFT codes (this will take a few minutes)

In [1]:
!pip install -q condacolab

In [2]:
import condacolab
condacolab.install()

✨🍰✨ Everything looks OK!


In [3]:
import subprocess
import sys
subprocess.run("rm -rf /usr/local/conda-meta/pinned", shell=True)
subprocess.run("pip -q install py3Dmol", shell=True)
subprocess.run("!mamba install -c anaconda intel-openmp", shell=True)
subprocess.run("conda config --add channels http://conda.anaconda.org/psi4", shell=True)
subprocess.run("mamba install psi4 resp -c conda-forge/label/libint_dev -c conda-forge", shell=True)
subprocess.run("pip install rdkit-pypi", shell=True)
subprocess.run("pip install Cython", shell=True)
subprocess.run("mamba install -c conda-forge parmed -y", shell=True)
subprocess.run("mamba install -c conda-forge openbabel -y", shell=True)
subprocess.run("pip install ase", shell=True)

CompletedProcess(args='pip install ase', returncode=0)

### Instructions
Use the __ase.build.molecule__ routine to create models of the acids and delete one atom to create the congugate base.

In [6]:
from ase.build import molecule
from ase import Atoms

acid = molecule('HCOOH')
print(acid.get_positions())
print(acid.get_atomic_numbers())
print(acid.get_chemical_formula())

base = molecule('HCOOH')
del base[3]
base.set_initial_charges([-1,0,0,0])
print(base.get_positions())
print(base.get_atomic_numbers())
print(base.get_chemical_formula())

molecules = [acid, base]

#code goes here

[[-1.040945 -0.436432  0.      ]
 [ 0.        0.423949  0.      ]
 [ 1.169372  0.103741  0.      ]
 [-0.64957  -1.335134  0.      ]
 [-0.377847  1.452967  0.      ]]
[8 6 8 1 1]
CH2O2
[[-1.040945 -0.436432  0.      ]
 [ 0.        0.423949  0.      ]
 [ 1.169372  0.103741  0.      ]
 [-0.377847  1.452967  0.      ]]
[8 6 8 1]
CHO2


In [16]:
#initalise dictionaries to save the results below
acid_results = {}
base_results = {}
h_results = {}

Optimise each of the molecules using Psi4 at the _B3LYP/3-21G_ level of theory *with* the BFGS optimiser to maximum force of <0.01 eV/Å.

In [10]:
from ase.calculators.psi4 import Psi4
from ase.build import molecule
import numpy as np
from ase.optimize import BFGS

optimised_molecules = []
for atoms in molecules:
  print(atoms.get_chemical_formula())
  calc = Psi4(atoms = atoms,
        method = 'b3lyp',
        memory = '1000MB', # this is the default, be aware!
        basis = '3-21G',

        charge = np.sum(atoms.get_initial_charges()),
        multiplicity=1,
        label = atoms.get_chemical_formula())

  atoms.calc = calc

  BFGS(atoms).run(fmax=0.01)

  #save results
  if atoms.get_chemical_formula() == 'CH2O2':
      acid_results['energy'] = atoms.get_potential_energy()
  if atoms.get_chemical_formula() == 'CHO2':
      base_results['energy'] = atoms.get_potential_energy()
  print(atoms.get_potential_energy())
  optimised_molecules.append(atoms)

CH2O2
      Step     Time          Energy         fmax
BFGS:    0 06:56:43    -5134.713234        0.0076
-5134.713233925359
CHO2
      Step     Time          Energy         fmax
BFGS:    0 06:56:46    -5118.546951        0.0098
-5118.546950586458


For each of the optimised molecules compute the vibrational and thermodynamic properties using _ase.vibrations_ and _ase.thermochemistry_ codes and save the results to the dictionaries _base\_results_ and _acid\_results_

For example:
```python
from ase.build import molecule
from ase.calculators.emt import EMT
from ase.optimize import QuasiNewton
from ase.vibrations import Vibrations
from ase.thermochemistry import IdealGasThermo

atoms = molecule('N2')
atoms.calc = Psi4()
potentialenergy = atoms.get_potential_energy()

vib = Vibrations(atoms)
vib.run()
vib_energies = vib.get_energies()

thermo = IdealGasThermo(vib_energies=vib_energies,
                        potentialenergy=potentialenergy,
                        atoms=atoms,
                        geometry='linear',
                        symmetrynumber=2, spin=0)
G = thermo.get_gibbs_energy(temperature=298.15, pressure=101325.)
```

In [17]:
from ase.vibrations import Vibrations
#save the results in a list for later processing
vibs = []
for atoms in optimised_molecules:
  print(atoms.get_chemical_formula())
  vib = Vibrations(atoms, name=atoms.get_chemical_formula(), delta=0.01)
  vib.clean()
  vib.run()
  vib.write_mode(0)
  vib.write_mode(-1)
  vib.summary()
  vibs.append(vib)

  from ase.thermochemistry import IdealGasThermo
# the 0th entry is acid
atoms = optimised_molecules[0]
potentialenergy = atoms.get_potential_energy()
vib = vibs[0]
vib_energies = vib.get_energies()


thermo = IdealGasThermo(vib_energies=vib_energies,
                        potentialenergy=potentialenergy,
                        atoms=atoms,
                        geometry='nonlinear',
                        symmetrynumber=1, spin=0)
G = thermo.get_gibbs_energy(temperature=298.15, pressure=101325.)
H = thermo.get_enthalpy(temperature=298.15, verbose=False)
S = thermo.get_entropy(temperature=298.15, pressure=101325., verbose=False)
acid_results['gibbs_energy'] = G
acid_results['enthalpy'] = H
acid_results['entropy'] = S

from ase.thermochemistry import IdealGasThermo
# the 1th entry is base
atoms = optimised_molecules[1]
potentialenergy = atoms.get_potential_energy()
vib = vibs[1]
vib_energies = vib.get_energies()


thermo = IdealGasThermo(vib_energies=vib_energies,
                        potentialenergy=potentialenergy,
                        atoms=atoms,
                        geometry='nonlinear',
                        symmetrynumber=1, spin=0)
G = thermo.get_gibbs_energy(temperature=298.15, pressure=101325.)
H = thermo.get_enthalpy(temperature=298.15, verbose=False)
S = thermo.get_entropy(temperature=298.15, pressure=101325., verbose=False)
base_results['gibbs_energy'] = G
base_results['enthalpy'] = H
base_results['entropy'] = S

from ase.thermochemistry import IdealGasThermo
# the - entry is

potentialenergy = 0.0
vib_energies = []

hplus = molecule('H')
hplus.set_initial_charges([+1])
atoms = hplus

thermo = IdealGasThermo(vib_energies=vib_energies,
                        potentialenergy=potentialenergy,
                        atoms=atoms,
                        geometry='monatomic',
                        symmetrynumber=1, spin=0)
G = thermo.get_gibbs_energy(temperature=298.15, pressure=101325.)
H = thermo.get_enthalpy(temperature=298.15, verbose=False)
S = thermo.get_entropy(temperature=298.15, pressure=101325., verbose=False)
h_results['gibbs_energy'] = G
h_results['enthalpy'] = H
h_results['entropy'] = S

CH2O2
---------------------
  #    meV     cm^-1
---------------------
  0    1.5i     12.3i
  1    1.3i     10.7i
  2    0.2i      1.5i
  3    0.1       0.6
  4    0.2       1.8
  5    1.9      15.4
  6   75.4     608.2
  7   85.4     688.5
  8  132.0    1064.5
  9  133.1    1073.6
 10  164.7    1328.6
 11  176.8    1425.8
 12  218.9    1765.8
 13  383.1    3089.5
 14  427.7    3449.3
---------------------
Zero-point energy: 0.900 eV
CHO2
---------------------
  #    meV     cm^-1
---------------------
  0    1.0i      8.2i
  1    1.0i      8.1i
  2    0.2i      1.5i
  3    0.2i      1.5i
  4    0.0i      0.3i
  5    3.9      31.3
  6   94.5     762.2
  7  133.0    1072.7
  8  160.7    1295.9
  9  179.8    1450.3
 10  214.8    1732.7
 11  290.9    2345.9
---------------------
Zero-point energy: 0.539 eV
Enthalpy components at T = 298.15 K:
E_pot              -5134.713 eV
E_ZPE                  0.898 eV
Cv_trans (0->T)        0.039 eV
Cv_rot (0->T)          0.039 eV
Cv_vib (0->T)      

<ipython-input-17-5851d0ba027b>:29: DeprecationWarning: `product` is deprecated as of NumPy 1.25.0, and will be removed in NumPy 2.0. Please use `prod` instead.
  S = thermo.get_entropy(temperature=298.15, pressure=101325., verbose=False)


Enthalpy components at T = 298.15 K:
E_pot              -5118.547 eV
E_ZPE                  0.537 eV
Cv_trans (0->T)        0.039 eV
Cv_rot (0->T)          0.039 eV
Cv_vib (0->T)          0.004 eV
(C_v -> C_p)           0.026 eV
-------------------------------
H                  -5117.904 eV

Entropy components at T = 298.15 K and P = 101325.0 Pa:
                           S               T*S
S_trans (1 bar)    0.0016203 eV/K        0.483 eV
S_rot              0.0008986 eV/K        0.268 eV
S_elec             0.0000000 eV/K        0.000 eV
S_vib              0.0000155 eV/K        0.005 eV
S (1 bar -> P)    -0.0000011 eV/K       -0.000 eV
-------------------------------------------------
S                  0.0025332 eV/K        0.755 eV

Free energy components at T = 298.15 K and P = 101325.0 Pa:
    H      -5117.904 eV
 -T*S         -0.755 eV
-----------------------
    G      -5118.659 eV
Enthalpy components at T = 298.15 K:
E_pot                  0.000 eV
E_ZPE                  0.00

<ipython-input-17-5851d0ba027b>:49: DeprecationWarning: `product` is deprecated as of NumPy 1.25.0, and will be removed in NumPy 2.0. Please use `prod` instead.
  S = thermo.get_entropy(temperature=298.15, pressure=101325., verbose=False)


At this point you should have three dictionaries _h\_results_, base\_results_ and _acid\_results_ that includes the electronic energy, enthalpy and entropy of these species.

__Calculate the change in internal energy $\Delta$U, the change in enthalpy $\Delta$H, the entropy contribution T$\Delta$S, and the Gibbs free energy change $\Delta$G for the reaction (Eqns 23-27) and compare to your results for water.__


In [20]:
## products - reactants
deltaH = (h_results['enthalpy'] + base_results['enthalpy']) - acid_results['enthalpy']
print (deltaH)

15.86313126713685


In [21]:
## products - reactants
T = 298.15

TdeltaS = T * ((h_results['entropy'] + base_results['entropy']) - acid_results['entropy'])
print (TdeltaS)

0.3226112911756765


In [22]:
#answer for deltaG
deltaG = (h_results['gibbs_energy'] + base_results['gibbs_energy']) - acid_results['gibbs_energy']
print (deltaG)

15.540519975961615


_Your discussion_

__Calculate K$_a$ and pK$_a$ at 298.15 K for the acid-ionisation reaction. Discuss how this compares with the experimentally determined value and why it might differ.__

Experimental values at 298 K can be found in the NIST Chemistry WebBook (https://webbook.nist.gov/chemistry/). In the "Gas phase ion energetics data" section, search under for "De- protonation Reactions" for the acid, or "Gas basicity" for the conjugate base.

In [23]:
import math

G = deltaG * (96.485 * 1000) ## in joules
T = 298.15
value = -(G // (8.314472 * T))
Ka = math.exp(value)
print(Ka)


4.8543706176772776e-263


In [24]:
pka = -math.log(Ka)
print(pka)

604.0


_Your discussion_